# A3-M3-2 · Chroma 저장과 검색
교안의 같은 번호 단원과 함께 봅니다. 노트북은 01 → 02 → 03 순서로 실행합니다.

## 01-1. CSV 읽고 FAQ 20건을 벡터로

In [1]:
import pandas as pd
from emb import embed

faq = pd.read_csv("shop_faq.csv")
faq["doc"] = faq["question"] + " " + faq["answer"]
X = embed(faq["doc"].tolist())                 # 문서는 passage
print(faq.shape, "→ 벡터", X.shape)
print(faq[["faq_id", "category", "question"]].head(3).to_string(index=False))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1438.16it/s]


(20, 5) → 벡터 (20, 384)
 faq_id category                question
    101       배송             배송은 언제 오나요?
    102       배송             배송비는 얼마인가요?
    103       배송 주문한 상품의 배송 조회는 어디서 하나요?


## 01-2. Chroma 클라이언트와 컬렉션 만들기

In [2]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")        # 이 폴더에 자동 저장
col = client.get_or_create_collection(
    name="shop_faq",
    configuration={"hnsw": {"space": "cosine"}},            # 거리: 코사인
)
print("컬렉션:", col.name, "| 저장된 개수:", col.count())
print("전체 컬렉션:", [c.name for c in client.list_collections()])

컬렉션: shop_faq | 저장된 개수: 0
전체 컬렉션: ['shop_faq']


## 01-3. add — id·문서·메타데이터·벡터를 한 번에 넣기

In [3]:
col.add(
    ids=[str(i) for i in faq["faq_id"]],                      # id는 문자열
    documents=faq["doc"].tolist(),                            # 원문
    metadatas=faq[["category", "question"]].to_dict("records"),   # 필터에 쓸 정보
    embeddings=X,                                             # 우리가 만든 e5 벡터
)
print("저장된 개수:", col.count())

저장된 개수: 20


## 01-4. get — 넣은 것 꺼내 보기

In [ ]:
r = col.get(ids=["101", "401"])
for i, m, d in zip(r["ids"], r["metadatas"], r["documents"]):
    print(i, m, "|", d[:30], "...")

r = col.get(where={"category": "환불"})
print("환불 카테고리:", r["ids"])

## 01-5. query — 질문과 가까운 FAQ 찾기

In [ ]:
q = embed(["택배 언제 와요?"], kind="query")       # 질문은 query
r = col.query(query_embeddings=q, n_results=3)
print("결과 키:", list(r.keys()))
print("ids:", r["ids"])
print("distances:", [[round(d, 3) for d in r["distances"][0]]])

for i, d, m in zip(r["ids"][0], r["distances"][0], r["metadatas"][0]):
    print(f"{i}  거리 {d:.3f}  유사도 {1 - d:.3f}  [{m['category']}] {m['question']}")

## 01-6. 결과를 표로 돌려주는 search() 함수

In [ ]:
def search(question, k=3, where=None):
    r = col.query(query_embeddings=embed([question], kind="query"), n_results=k, where=where)
    return pd.DataFrame({
        "faq_id": [int(i) for i in r["ids"][0]],
        "score": [round(1 - d, 3) for d in r["distances"][0]],
        "category": [m["category"] for m in r["metadatas"][0]],
        "question": [m["question"] for m in r["metadatas"][0]],
    })

print(search("환불 며칠 걸려요?").to_string(index=False))